# 5.5 — MuscleMimic Halloween Dodge

Train a compact command-conditioned MyoFullBody controller for the Halloween pumpkin-dodge web demo, export it to ONNX, and copy the matching metadata into `docs/halloween_demo/model/`.

The browser and this notebook share one fixed 8-value dodge command: `escape_x, escape_y, duck, torso_lean_x, torso_lean_y, urgency, time_to_contact, threat_height`. Both manual and automatic pumpkin throws are converted to this command by the browser threat predictor.

This first implementation uses behavior cloning / distillation from MuscleMimic-style reference motions rather than training obstacle avoidance end-to-end. The web interface therefore remains independent of how the student policy is trained.

In [ ]:
from __future__ import annotations
import json, math, os, random
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import torch
from torch import nn

ROOT = Path.cwd()
if not (ROOT / 'myosuite').exists() and (ROOT.parent / 'myosuite').exists():
    ROOT = ROOT.parent
OUT = ROOT / 'docs' / 'halloween_demo' / 'model'
OUT.mkdir(parents=True, exist_ok=True)
ACTION_DIM = 354
COMMAND_DIM = 8
CONTROL_HZ = 50
print('output:', OUT)

## 1. Shared command contract

Keep this ordering synchronized with `docs/halloween_demo/web/threat_predictor.js`.

In [ ]:
COMMAND_NAMES = [
    'escape_x', 'escape_y', 'duck', 'torso_lean_x',
    'torso_lean_y', 'urgency', 'time_to_contact', 'threat_height'
]
assert len(COMMAND_NAMES) == COMMAND_DIM

def sample_command(rng=np.random.default_rng()):
    side = rng.choice([-1.0, 1.0])
    ttc = rng.uniform(0.25, 1.2)
    h = rng.uniform(0.45, 1.8)
    urgency = np.clip(1.0 - ttc / 1.2, 0.0, 1.0)
    duck = urgency if h > 1.15 else 0.0
    return np.array([0.0, -side, duck, 0.0, -0.45 * side, urgency, ttc, h], np.float32)

sample_command()

## 2. Build the student dataset

For the first web-capable policy we collect state/action supervision from a small set of permissively redistributable dodge primitives (neutral, crouch, lean L/R, sidestep L/R, recover). The collection hook below is deliberately separated from training so we can replace it later with stronger RL or MuscleMimic teacher rollouts without changing the browser model contract.

Expected arrays: `state[N,S]`, `command[N,8]`, `action[N,354]`.

In [ ]:
DATA = ROOT / 'runs' / 'halloween_dodge' / 'student_dataset.npz'
DATA.parent.mkdir(parents=True, exist_ok=True)

def load_dataset(path=DATA):
    if not path.exists():
        raise FileNotFoundError(
            f'{path} not found. Collect teacher/reference rollouts first. '
            'The dataset must contain state, command and action arrays.'
        )
    d = np.load(path)
    state = d['state'].astype(np.float32)
    command = d['command'].astype(np.float32)
    action = d['action'].astype(np.float32)
    assert command.shape[1] == COMMAND_DIM
    assert action.shape[1] == ACTION_DIM
    return state, command, action


## 3. Command-conditioned student

The student intentionally uses a small MLP that exports cleanly to ONNX Runtime Web. Normalization lives in the exported metadata rather than being hidden in notebook code.

In [ ]:
class DodgeStudent(nn.Module):
    def __init__(self, input_dim, action_dim=ACTION_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ELU(),
            nn.Linear(512, 512), nn.ELU(),
            nn.Linear(512, 256), nn.ELU(),
            nn.Linear(256, action_dim), nn.Tanh(),
        )
    def forward(self, x):
        return self.net(x)

def train_student(state, command, action, epochs=30, batch_size=2048, lr=3e-4):
    x = np.concatenate([state, command], axis=1).astype(np.float32)
    mean = x.mean(0)
    std = np.maximum(x.std(0), 1e-6)
    xn = (x - mean) / std
    model = DodgeStudent(x.shape[1])
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    tx = torch.from_numpy(xn)
    ty = torch.from_numpy(action)
    for epoch in range(epochs):
        order = torch.randperm(len(tx))
        total = 0.0
        for i in range(0, len(tx), batch_size):
            idx = order[i:i+batch_size]
            pred = model(tx[idx])
            loss = loss_fn(pred, ty[idx])
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(idx)
        print(f'epoch {epoch+1:03d} mse={total/len(tx):.6f}')
    return model.eval(), mean.astype(np.float32), std.astype(np.float32)


## 4. Train


In [ ]:
# Uncomment after collection:
# state, command, action = load_dataset()
# model, obs_mean, obs_std = train_student(state, command, action)


## 5. Export ONNX + web metadata

This is the only artifact interface the browser depends on.

In [ ]:
def export_web_policy(model, obs_mean, obs_std, state_dim, out_dir=OUT):
    out_dir.mkdir(parents=True, exist_ok=True)
    input_dim = state_dim + COMMAND_DIM
    onnx_path = out_dir / 'halloween_dodge.onnx'
    dummy = torch.zeros(1, input_dim, dtype=torch.float32)
    torch.onnx.export(
        model, dummy, onnx_path,
        input_names=['obs'], output_names=['action'],
        dynamic_axes={'obs': {0: 'batch'}, 'action': {0: 'batch'}},
        opset_version=17,
    )
    meta = {
        'format_version': 1,
        'model': 'MyoFullBody',
        'action_dim': ACTION_DIM,
        'state_dim': int(state_dim),
        'command_dim': COMMAND_DIM,
        'input_dim': int(input_dim),
        'command_names': COMMAND_NAMES,
        'obs_mean': obs_mean.tolist(),
        'obs_std': obs_std.tolist(),
        'control_hz': CONTROL_HZ,
        'onnx_file': 'halloween_dodge.onnx',
    }
    (out_dir / 'halloween_dodge.json').write_text(json.dumps(meta, indent=2))
    print('wrote', onnx_path)
    return onnx_path, meta

# Example after training:
# export_web_policy(model, obs_mean, obs_std, state.shape[1])

## 6. ONNX parity check


In [ ]:
def verify_onnx(model, obs_mean, obs_std, state_dim, onnx_path=OUT/'halloween_dodge.onnx'):
    import onnxruntime as ort
    x = np.random.randn(32, state_dim + COMMAND_DIM).astype(np.float32)
    xn = (x - obs_mean) / obs_std
    with torch.no_grad():
        ref = model(torch.from_numpy(xn)).cpu().numpy()
    sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
    got = sess.run(['action'], {'obs': xn})[0]
    err = np.max(np.abs(ref - got))
    print('max |PyTorch-ONNX| =', err)
    assert err < 1e-4
    return err


## Next collection milestone

Wire MyoFullBody/MuscleMimic teacher rollouts into `student_dataset.npz`. We should start with 6 dodge primitives and randomized transition phases, then evaluate held-out pumpkin trajectories. Once that is robust, the same notebook can be upgraded to RL fine-tuning without changing the exported browser contract.